# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading and exploring the FAIR<sup>2</sup> colorectal cancer dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library, based on its Croissant metadata schema.

### Dataset Source
The dataset is described by a Croissant schema available [here](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).


In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")


## 2. Data Overview

Let's review the available record sets, their `@id`s, and fields. We'll also display field `@id`s and names to see what data is available.


In [ ]:
# Retrieve all record sets from the dataset by @id
record_sets = []
record_sets_meta = []
if hasattr(metadata, 'record_set') and metadata.record_set:
    for rs in metadata.record_set:
        record_sets.append(rs['@id'])
        record_sets_meta.append(rs)
else:
    # Try to get record sets from the dataset.fields method (for some datasets)
    if hasattr(dataset, 'record_sets'):
        for rs in dataset.record_sets:
            record_sets.append(rs['@id'])
            record_sets_meta.append(rs)

print(f"Available record sets by @id:\n", record_sets)

# Display record set fields and their @ids
from mlcroissant.types.field import Field

for rs_id in record_sets:
    print(f"\nRecord Set: {rs_id}")
    fields = dataset.fields(record_set=rs_id)
    for f in fields:
        print(f"  Field: {f['@id']} | name: {f.get('name', f.get('@id'))}")

## 3. Data Extraction

Let's load data from the main record set(s) into pandas DataFrames. We'll use the record set and its field `@id`s as reference, as listed above.

**Note:** The FAIR² CRC dataset may contain a single main record set depending on schema, but we support multiple for generality.

In [ ]:
# Load records from each record set into pandas DataFrames
dfs = dict()

for rs_id in record_sets:
    print(f"Loading data for record set: {rs_id}")
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"{len(df)} records loaded. Columns: {list(df.columns)}\n")

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic exploratory analysis on a numeric field. We'll select a numeric field from the record set, filter for values above a threshold, normalize values, and group by a key field if available.


In [ ]:
# Choose a record set and fields for EDA
if record_sets:
    selected_rs_id = record_sets[0]
    df = dfs[selected_rs_id]
else:
    raise ValueError('No record sets found in dataset')

print(f"Sample records from record set {selected_rs_id}:")
display(df.head())

# Infer or guess a numeric field (@id) likely to be present.
# We'll look for an integer/float column (common ones in such datasets: age, interval, etc.)
import numpy as np
numeric_field_id = None
for col in df.columns:
    if np.issubdtype(df[col].dropna().dtype, np.number):
        numeric_field_id = col
        break

if numeric_field_id is None:
    print('No numeric field detected.')
    numeric_field_id = df.columns[0]  # fallback

print(f"Using numeric field (column @id): {numeric_field_id}")

# Thresholding
threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 10
filtered_df = df[df[numeric_field_id] > threshold] if np.issubdtype(df[numeric_field_id].dtype, np.number) else df.copy()
print(f"Filtered to records where {numeric_field_id} > {threshold} (if numeric). Remaining: {len(filtered_df)}\n")

# Normalization
if np.issubdtype(df[numeric_field_id].dtype, np.number):
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group (categorical) field
group_field_id = None
for col in df.columns:
    if (df[col].dtype == object) and (df[col].nunique() < 10) and (col != numeric_field_id):
        group_field_id = col
        break

if group_field_id:
    print(f"\nGrouping by field @id: {group_field_id}")
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean','count']).reset_index()
    print(grouped.head())

## 5. Visualization

Visualize the distribution of the numeric field and its relation to possible groupings (e.g., bar plot by group/categorical field).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
if np.issubdtype(df[numeric_field_id].dtype, np.number):
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], kde=True, color='royalblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

# Barplot by group if available
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=df, ci=None)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, you explored the FAIR² colorectal cancer dataset via its Croissant schema and the `mlcroissant` library. You:
- Loaded the dataset and discovered all available record sets and their schema-defined field `@id`s.
- Extracted tabular data from each record set into pandas DataFrames using their `@id` references.
- Performed basic EDA including thresholding, normalization, and grouping by categorical fields using only entity `@id`s.
- Visualized distributions using histograms and barplots.

You can now further analyze, filter, or visualize this dataset as needed for more specific clinical/statistical research questions!